# Criteo Train

Minimal path to train an XGBoost CTR model: load + encode data, then either a single fit or a wandb hyperparameter sweep.

For exploratory analysis, frequent-itemset/association-rule mining, and tree inspection, see `criteo_analysis.ipynb`.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import polars as pl
import wandb
import xgboost as xgb
from sklearn.metrics import log_loss
from wandb.integration.xgboost import WandbCallback

from criteo_functions import *

## Load data

In [ ]:
data_dir = Path("../data/Criteo")

config = {
    "sample_every": 1,
    "cat_cutoff": 25,
    "cat_pseudo_count": 16,
    "data_dir": data_dir,
}

train_X, train_y, valid_X, valid_y, test_X, test_y = get_data(config)
train_X.shape, valid_X.shape, test_X.shape

## Train (single fit)

Hyperparameters below are the best found in the `criteo_boost_sweep` sweep (see `wandb.md`): `learning_rate=0.1`, `max_depth=60`, `subsample=1.0`, `gamma=10`, `reg_lambda=0`.

In [ ]:
with wandb.init(project="criteo_boost", config=config) as run:
    clf = xgb.XGBClassifier(
        tree_method="hist",
        early_stopping_rounds=10,
        reg_lambda=0,
        gamma=10,
        subsample=1.0,
        n_estimators=150,
        learning_rate=0.1,
        max_depth=60,
        callbacks=[WandbCallback(log_feature_importance=True)],
    )
    clf.fit(
        train_X, train_y,
        eval_set=[(train_X, train_y), (valid_X, valid_y)],
        verbose=2,
    )

In [ ]:
log_loss(test_y, clf.predict_proba(test_X))

## Train (GPU)

In [ ]:
xgb.build_info()  # confirm CUDA/GPU support is compiled in

In [ ]:
with wandb.init(project="criteo_boost_gpu", config=config) as run:
    clf_gpu = xgb.XGBClassifier(
        device="gpu",
        tree_method="hist",
        early_stopping_rounds=10,
        reg_lambda=0,
        gamma=10,
        subsample=1.0,
        n_estimators=150,
        learning_rate=0.1,
        max_depth=60,
        callbacks=[WandbCallback(log_feature_importance=True)],
    )
    clf_gpu.fit(
        train_X, train_y,
        eval_set=[(train_X, train_y), (valid_X, valid_y)],
        verbose=2,
    )

In [ ]:
log_loss(test_y, clf_gpu.predict_proba(test_X))

## Hyperparameter sweep

`train()` below is the wandb sweep-agent entry point: it reads hyperparameters from
`wandb.config`, fits an `XGBClassifier`, and logs to the run started by `wandb.agent`.

In [ ]:
def train():
    """wandb sweep agent entry point: trains an XGBClassifier for one sweep run."""
    config = {
        'sample_every': 1,
        'cat_cutoff': 25,
        'cat_pseudo_count': 16,
        'data_dir': Path("../data/Criteo"),
        'tree_method': "hist",
        'early_stopping_rounds': 10,
        'reg_lambda': 0,
        'gamma': 10,
        'n_estimators': 150,
        'learning_rate': .1,
        'max_depth': 60,
    }
    data_keys = ('sample_every', 'cat_cutoff', 'cat_pseudo_count', 'data_dir')

    with wandb.init(config=config) as run:
        config_data = {key: value for key, value in wandb.config.items() if key in data_keys}
        config_xgb = {key: value for key, value in wandb.config.items() if key not in data_keys}
        train_X, train_y, valid_X, valid_y, _test_X, _test_y = get_data(config_data)
        clf = xgb.XGBClassifier(**config_xgb, callbacks=[WandbCallback(log_feature_importance=True)])
        clf.fit(train_X, train_y, eval_set=[(train_X, train_y), (valid_X, valid_y)], verbose=2,)
        tree_df = clf.get_booster().trees_to_dataframe()

In [ ]:
sweep_config = {
    "method": "grid",  # try grid or random
    "metric": {
        "name": "validation_1-logloss",
        "goal": "minimize",
    },
    "parameters": {
        "learning_rate": {"values": [0.1]},
        "gamma": {"values": [0, 5, 10, 15]},
        "reg_lambda": {"values": [0, 1, 10]},
        "min_child_weight": {"values": [1, 2, 4, 8, 16]},
        "max_depth": {"values": [50, 55, 60, 65, 70, 75, 80]},
    },
}

In [ ]:
sweep_id = wandb.sweep(sweep_config, project="criteo_boost_sweep")

In [ ]:
wandb.agent(sweep_id, train)